# CourtIQ — Custom YOLOX Basketball Detector Training

Trains a **YOLOX-S** model (Apache 2.0 — fully commercial) to detect:
- **basketball** (the ball)
- **basketball-hoop** (rim + backboard)

Using the HotShot basketball detection dataset (7.4k images) from Roboflow Universe.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Run all cells in order
3. Download the trained ONNX model from the last cell

## Step 1: Install Dependencies

In [ ]:
!pip install roboflow -q

# Clone YOLOX repo
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX
!pip install -v -e . -q
!pip install cython_bbox pycocotools -q

## Step 2: Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU")

## Step 3: Download Dataset from Roboflow (COCO format)

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="35nWQZay8j7BoeXMy2hK")
project = rf.workspace("hotshot").project("basketball-detection-tqwcs")
version = project.version(7)
dataset = version.download("coco")

DATASET_DIR = dataset.location
print(f"Dataset location: {DATASET_DIR}")

# Check structure
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split)
    if os.path.isdir(img_dir):
        count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))])
        print(f"  {split}: {count} images")

## Step 4: Create YOLOX Experiment Config

In [ ]:
import json

# Read COCO annotations to get class info
with open(os.path.join(DATASET_DIR, 'train', '_annotations.coco.json'), 'r') as f:
    coco = json.load(f)

categories = coco['categories']
num_classes = len(categories)
class_names = [c['name'] for c in sorted(categories, key=lambda x: x['id'])]

print(f"Classes ({num_classes}): {class_names}")
print(f"Training images: {len(coco['images'])}")
print(f"Training annotations: {len(coco['annotations'])}")

In [ ]:
# Create custom YOLOX experiment file
exp_content = f'''
import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        # Model
        self.depth = 0.33    # YOLOX-S
        self.width = 0.50    # YOLOX-S
        self.num_classes = {num_classes}

        # Training
        self.max_epoch = 100
        self.warmup_epochs = 5
        self.basic_lr_per_img = 0.01 / 64.0
        self.no_aug_epochs = 15
        self.eval_interval = 5
        self.print_interval = 50

        # Data
        self.input_size = (416, 416)
        self.test_size = (416, 416)
        self.data_num_workers = 2

        # Augmentation
        self.mosaic_prob = 1.0
        self.mixup_prob = 0.5
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 10.0
        self.translate = 0.1
        self.mosaic_scale = (0.5, 1.5)
        self.mixup_scale = (0.5, 1.5)
        self.enable_mixup = True

        self.exp_name = "courtiq_basketball"
        self.output_dir = "YOLOX_outputs"

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=False):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection
        import torch

        dataset = COCODataset(
            data_dir="{DATASET_DIR}/train",
            json_file="_annotations.coco.json",
            img_size=self.input_size,
            preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            cache=False,
        )

        dataset = MosaicDetection(
            dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob if self.enable_mixup else 0.0,
            enable_mixup=self.enable_mixup,
        )

        sampler = InfiniteSampler(len(dataset), seed=0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )

        dataloader = DataLoader(dataset, batch_sampler=batch_sampler, num_workers=self.data_num_workers, pin_memory=True)
        return dataloader

    def get_eval_loader(self, batch_size, is_distributed, testdev=False, legacy=False):
        from yolox.data import COCODataset, ValTransform
        import torch

        valdataset = COCODataset(
            data_dir="{DATASET_DIR}/valid",
            json_file="_annotations.coco.json",
            img_size=self.test_size,
            preproc=ValTransform(legacy=legacy),
            cache=False,
        )

        sampler = torch.utils.data.SequentialSampler(valdataset)
        dataloader = torch.utils.data.DataLoader(valdataset, batch_size=batch_size, sampler=sampler, num_workers=self.data_num_workers, pin_memory=True)
        return dataloader

    def get_evaluator(self, batch_size, is_distributed, testdev=False, legacy=False):
        from yolox.evaluators import COCOEvaluator

        val_loader = self.get_eval_loader(batch_size, is_distributed, testdev, legacy)
        evaluator = COCOEvaluator(
            dataloader=val_loader,
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )
        return evaluator
'''

with open('courtiq_exp.py', 'w') as f:
    f.write(exp_content)

print("Created courtiq_exp.py")

## Step 5: Download Pretrained YOLOX-S Weights

In [ ]:
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth
print(f"Downloaded yolox_s.pth ({os.path.getsize('yolox_s.pth')/1024/1024:.1f} MB)")

## Step 6: Train YOLOX-S

Training config:
- Model: YOLOX-S (small — good for browser inference, Apache 2.0 license)
- Epochs: 100
- Image size: 416×416 (matches CourtIQ inference)
- Batch: 16

In [ ]:
!python tools/train.py \
    -f courtiq_exp.py \
    -c yolox_s.pth \
    -d 1 \
    -b 16 \
    --fp16 \
    -o \
    --cache

## Step 7: Evaluate Results

In [ ]:
# Find best checkpoint
import glob

checkpoints = sorted(glob.glob('YOLOX_outputs/courtiq_basketball/**/best_ckpt.pth', recursive=True))
if checkpoints:
    best_ckpt = checkpoints[-1]
    print(f"Best checkpoint: {best_ckpt}")
    print(f"Size: {os.path.getsize(best_ckpt)/1024/1024:.1f} MB")
else:
    # Fallback to latest
    checkpoints = sorted(glob.glob('YOLOX_outputs/courtiq_basketball/**/latest_ckpt.pth', recursive=True))
    best_ckpt = checkpoints[-1] if checkpoints else None
    print(f"Using latest checkpoint: {best_ckpt}")

In [ ]:
# Run evaluation
!python tools/eval.py \
    -f courtiq_exp.py \
    -c {best_ckpt} \
    -d 1 \
    -b 16 \
    --conf 0.25 \
    --fp16

## Step 8: Export to ONNX for Browser

Export to ONNX format for use with ONNX Runtime Web.

In [ ]:
!python tools/export_onnx.py \
    --output-name basketball_yolox_courtiq.onnx \
    -f courtiq_exp.py \
    -c {best_ckpt} \
    --input 416 416 \
    --opset 11

onnx_path = "basketball_yolox_courtiq.onnx"
print(f"\nONNX model: {onnx_path}")
print(f"Size: {os.path.getsize(onnx_path)/1024/1024:.1f} MB")

## Step 9: Verify ONNX Model Shape

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)

print("=== INPUT ===")
for inp in session.get_inputs():
    print(f"  Name: {inp.name}")
    print(f"  Shape: {inp.shape}")
    print(f"  Type: {inp.type}")

print("\n=== OUTPUT ===")
for out in session.get_outputs():
    print(f"  Name: {out.name}")
    print(f"  Shape: {out.shape}")
    print(f"  Type: {out.type}")

print(f"\nClass names: {class_names}")
print(f"\nNOTE: Update ai-shot-tracker.js class indices to match:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## Step 10: Quick Inference Test

In [ ]:
import cv2
import numpy as np
from IPython.display import display
from PIL import Image

# Load a test image
test_images = glob.glob(f"{DATASET_DIR}/test/*.jpg")[:3]

for img_path in test_images:
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Preprocess: resize + pad to 416x416
    ratio = min(416/h, 416/w)
    new_h, new_w = int(h*ratio), int(w*ratio)
    resized = cv2.resize(img, (new_w, new_h))
    padded = np.full((416, 416, 3), 114, dtype=np.uint8)
    padded[:new_h, :new_w] = resized

    # To float, CHW, batch
    input_tensor = padded.astype(np.float32).transpose(2, 0, 1)[np.newaxis]

    # Run inference
    outputs = session.run(None, {session.get_inputs()[0].name: input_tensor})
    out = outputs[0][0]  # [num_detections, 7] or similar

    print(f"\n{os.path.basename(img_path)}: output shape {outputs[0].shape}")

    # YOLOX output: [batch, num_anchors, 5+num_classes]
    # Each row: [cx, cy, w, h, obj_conf, cls1_score, cls2_score, ...]
    for det in out:
        obj_conf = det[4]
        cls_scores = det[5:5+num_classes]
        cls_id = np.argmax(cls_scores)
        score = obj_conf * cls_scores[cls_id]
        if score > 0.3:
            print(f"  {class_names[cls_id]}: {score:.2f} (obj={obj_conf:.2f})")

## Step 11: Download Model

Download the ONNX model. Upload it to CourtIQ as `models/basketball_yolox.onnx`.

In [ ]:
from google.colab import files

print(f"Model: {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)")
print(f"Classes: {class_names}")
print("\nDownloading...")
files.download(onnx_path)

In [ ]:
# Also download the PyTorch checkpoint as backup
import shutil
shutil.copy(best_ckpt, "courtiq_basketball_best.pth")
files.download("courtiq_basketball_best.pth")